-word2vec
-Glove
-fasttext

In [ ]:
!pip install gensim

In [ ]:
import gensim.downloader as api

In [ ]:
api.info()

{'corpora': {'semeval-2016-2017-task3-subtaskBC': {'num_records': -1,
   'record_format': 'dict',
   'file_size': 6344358,
   'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/semeval-2016-2017-task3-subtaskB-eng/__init__.py',
   'license': 'All files released for the task are free for general research use',
   'fields': {'2016-train': ['...'],
    '2016-dev': ['...'],
    '2017-test': ['...'],
    '2016-test': ['...']},
   'description': 'SemEval 2016 / 2017 Task 3 Subtask B and C datasets contain train+development (317 original questions, 3,169 related questions, and 31,690 comments), and test datasets in English. The description of the tasks and the collected data is given in sections 3 and 4.1 of the task paper http://alt.qcri.org/semeval2016/task3/data/uploads/semeval2016-task3-report.pdf linked in section “Papers” of https://github.com/RaRe-Technologies/gensim-data/issues/18.',
   'checksum': '701ea67acd82e75f95e1d8e62fb0ad29',
   'file_name': 'se

In [ ]:
pretrained_word2vec_model = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
pretrained_word2vec_model['dog'] # Vector of a given word

In [ ]:
len(pretrained_word2vec_model['dog']) # Checking vector dimensions

300

In [ ]:
pretrained_word2vec_model.most_similar('dog')

[('dogs', 0.8680489659309387),
 ('puppy', 0.8106428384780884),
 ('pit_bull', 0.780396044254303),
 ('pooch', 0.7627376914024353),
 ('cat', 0.7609457969665527),
 ('golden_retriever', 0.7500901818275452),
 ('German_shepherd', 0.7465174198150635),
 ('Rottweiler', 0.7437615394592285),
 ('beagle', 0.7418621778488159),
 ('pup', 0.740691065788269)]

# Data acquisition

In [ ]:
import pandas as pd

In [ ]:
amazon_df = pd.read_csv('amazon_cells_labelled.txt', sep='\t', header=None)

In [ ]:
yelp_df = pd.read_csv('yelp_labelled.txt', sep='\t', header = None)

In [ ]:
imdb_df = pd.read_csv('imdb_labelled.txt', sep='\t', header = None)

In [ ]:
# Combining all the datasets into one big dataset
master_df = pd.concat([amazon_df, yelp_df, imdb_df])

In [ ]:
master_df

,0,1
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1
...,...,...
743,I just got bored watching Jessice Lange take h...,0
744,"Unfortunately, any virtue in this film's produ...",0
745,"In a word, it is embarrassing.",0
746,Exceptionally bad!,0


In [ ]:
master_df[1].value_counts()

,count
1,
1,1386
0,1362


# Text Cleaning & Preprocessing

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
def clean_preprocess(text):
    text = text.lower()
    doc = nlp(text)
    text = [token.text for token in doc if (not token.is_punct) and (not token.like_num)]
    preprocessed_text = [word for word in text if word not in ENGLISH_STOP_WORDS]
    #preprocessed_text = ' '.join(text)
    return preprocessed_text

In [ ]:
master_df[0] = master_df[0].apply(lambda row: clean_preprocess(row))

In [ ]:
master_df

,0,1
0,"[way, plug, unless, converter]",0
1,"[good, case, excellent, value]",1
2,"[great, jawbone]",1
3,"[tied, charger, conversations, lasting, minute...",0
4,"[mic, great]",1
...,...,...
743,"[just, got, bored, watching, jessice, lange, c...",0
744,"[unfortunately, virtue, film, 's, production, ...",0
745,"[word, embarrassing, ]",0
746,"[exceptionally, bad, ]",0


# Text Representation or Feature Engineering (using word2vec pretrained model)

In [ ]:
import numpy as np

def convert_word_to_vector(text):
  total_vector = []

  for row in text: # Iterating each row in the text column
    count = 0
    row_total_vector = np.zeros(300) # Initializing zeroes with the same size as vector
    for word in row: # Iterating over each word in a row
      if word in pretrained_word2vec_model:
        row_total_vector = row_total_vector + pretrained_word2vec_model[word]
        count = count + 1

    # This block needs to be inside the outer loop to append a vector for each row
    if count != 0:
      row_avg_vector = row_total_vector/count
      total_vector.append(row_avg_vector)
    else:
      total_vector.append(np.zeros(300))

  return total_vector

In [ ]:
X_vector = convert_word_to_vector(master_df[0])

In [ ]:
np.shape(X_vector)

(2748, 300)

In [ ]:
np.shape(master_df[1])

(2748,)

# Training the Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_vector, master_df[1], random_state=1)

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train) # Training the model

LogisticRegression()

# Evaluating the Model

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(y_pred, y_test)

0.8296943231441049

# Applying or using the model

In [ ]:
review = "I like this movie"
newreview = convert_word_to_vector([clean_preprocess(review)])

In [ ]:
sentiment_score = model.predict(newreview)

In [ ]:
sentiment_score

array([1])

### Saving the Model with Pickle

To save the trained model, we can use Python's built-in `pickle` module. This module implements binary protocols for serializing and de-serializing a Python object structure.

First, we'll import the `pickle` library. Then, we'll use `pickle.dump()` to write the `model` object to a file in binary write mode (`'wb'`).


In [ ]:
import pickle

# Save the trained model to a file
filename = 'logistic_regression_model.pkl'
pickle.dump(model, open(filename, 'wb'))

print(f"Model successfully saved as '{filename}'")


Model successfully saved as 'logistic_regression_model.pkl'
